###  Main problem statement: “To determine which client segments generate the highest gross profit while maintaining strong customer
### satisfaction” (enabling the company to prioritize high-value clients, improve client retention, and support
### sustainable business growth)​

#### Key points (subproblems)​

#### Profitability and its drivers by client segment: Analyse gross profit and gross margin across client type,
#### industry sector, organisation size and location, while examining hardware, software and manpower costs
#### and service ratings to identify high-value segments and opportunities for cost optimisation and margin
#### improvement.​

#### Satisfaction and its drivers by client segment: Analyse Presales & Partnership, Technical Expertise,
#### Project Delivery, Post-Sales Support and NPS across client segments to identify consistently high- or low-
#### satisfaction groups and determine which service dimensions are most strongly associated with customer
#### satisfaction and loyalty.​

In [1]:
import pandas as pd
import plotly.express as px

# Load and prepare data
xls = pd.ExcelFile('merged.xlsx')
df_merged = pd.read_excel(xls, xls.sheet_names[0])

df_merged['COGS'] = (
    df_merged['HARDWARE']
    + df_merged['SOFTWARE']
    + df_merged['MANPOWER']
)

df_merged['GROSS_PROFIT'] = df_merged['REVENUE'] - df_merged['COGS']
df_merged['GROSS_MARGIN'] = (
    df_merged['GROSS_PROFIT'] / df_merged['REVENUE']
) * 100

# Aggregate by Sector
profit_segment = (
    df_merged
    .groupby("SECTOR")
    .agg({
        "GROSS_PROFIT":"sum",
        "GROSS_MARGIN":"mean",
        "REVENUE":"sum",
        "NPS RATING":"mean"
    })
    .reset_index()
)

fig = px.scatter(
    profit_segment,
    x="GROSS_MARGIN",
    y="GROSS_PROFIT",
    size="REVENUE",
    color="SECTOR",
    hover_name="SECTOR",
    hover_data={
        "REVENUE":":,",
        "NPS RATING":":.2f",
        "GROSS_MARGIN":":.2f"
    },
    title="Gross Profit vs Gross Margin Across Industry Sectors",
    labels={
        "GROSS_MARGIN":"Average Gross Margin (%)",
        "GROSS_PROFIT":"Total Gross Profit"
    }
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

In [2]:
import plotly.graph_objects as go

corr_cols = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT",
    "NPS RATING"
]

corr = df_merged[corr_cols].corr()

fig = go.Figure(
    data=go.Heatmap(
        z=corr.values,
        x=corr.columns,
        y=corr.columns,
        text=round(corr,2).values,
        texttemplate="%{text}",
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Correlation")
    )
)

fig.update_layout(
    title="Correlation Between Service Ratings and NPS",
    template="plotly_white"
)

fig.show()

In [3]:
import plotly.graph_objects as go

service_cols = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT",
    "NPS RATING"
]
segmentations = {
    "Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

corr_matrices = {}

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)[service_cols]
        .mean()
    )

    corr_matrices[label] = grouped.corr()

default = "Sector"

corr = corr_matrices[default]

fig = go.Figure()

fig.add_trace(

    go.Heatmap(

        z=corr.values,
        x=corr.columns,
        y=corr.columns,

        colorscale="RdBu",
        zmin=-1,
        zmax=1,

        text=round(corr,2).values,
        texttemplate="%{text}",

        colorbar=dict(title="Correlation")

    )

)
buttons = []

for label in corr_matrices:

    corr = corr_matrices[label]

    buttons.append(

        dict(

            label=label,

            method="update",

            args=[

                {
                    "z":[corr.values],
                    "text":[round(corr,2).values],
                    "x":[corr.columns],
                    "y":[corr.columns]
                },

                {
                    "title":f"Correlation Heatmap ({label})"
                }

            ]

        )

    )
fig.update_layout(

    title="Correlation Heatmap (Sector)",

    updatemenus=[

        dict(

            buttons=buttons,

            direction="down",

            x=1.15,

            y=1.1,

            showactive=True

        )

    ],

    template="plotly_white"

)

fig.show()

In [4]:
import plotly.graph_objects as go
import pandas as pd
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

aggregated = {}

for label, col in segmentations.items():

    grouped = (
        df_merged
        .groupby(col)[["HARDWARE", "SOFTWARE", "MANPOWER"]]
        .sum()
        .reset_index()
    )

    aggregated[label] = grouped
default = "Industry Sector"

data = aggregated[default]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["HARDWARE"],
        name="Hardware"
    )
)

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["SOFTWARE"],
        name="Software"
    )
)

fig.add_trace(
    go.Bar(
        x=data["SECTOR"],
        y=data["MANPOWER"],
        name="Manpower"
    )
)
buttons = []

for label, column in segmentations.items():

    temp = aggregated[label]

    buttons.append(

        dict(

            label=label,

            method="update",

            args=[

                {
                    "x":[
                        temp[column],
                        temp[column],
                        temp[column]
                    ],

                    "y":[
                        temp["HARDWARE"],
                        temp["SOFTWARE"],
                        temp["MANPOWER"]
                    ]

                },

                {
                    "title":f"Cost Composition by {label}",
                    "xaxis":{"title":label}
                }

            ]

        )

    )
mode_buttons = [

    dict(

        label="Stacked",

        method="relayout",

        args=[{"barmode":"stack"}]

    ),

    dict(

        label="Grouped",

        method="relayout",

        args=[{"barmode":"group"}]

    )

]
fig.update_layout(

    title="Cost Composition by Industry Sector",

    xaxis_title="Industry Sector",

    yaxis_title="Total Cost",

    barmode="stack",

    template="plotly_white",

    updatemenus=[

        dict(

            buttons=buttons,

            direction="down",

            x=0.02,

            y=1.18,

            showactive=True

        ),

        dict(

            buttons=mode_buttons,

            direction="right",

            x=0.55,

            y=1.18,

            showactive=True

        )

    ]

)

fig.show()

In [6]:
import plotly.graph_objects as go
import pandas as pd

# -----------------------------------------
# Metrics available
# -----------------------------------------
metrics = {
    "Gross Profit": "GROSS_PROFIT",
    "Gross Margin": "GROSS_MARGIN",
    "NPS": "NPS RATING"
}

# -----------------------------------------
# Segmentations available
# -----------------------------------------
segmentations = {
    "Industry Sector": "SECTOR",
    "Client Type": "TYPE",
    "Organisation Size": "STAFF STRENGTH",
    "Country": "COUNTRY"
}

# -----------------------------------------
# Aggregate data
# -----------------------------------------
aggregated = {}

for seg_name, seg_col in segmentations.items():

    grouped = (
        df_merged
        .groupby(seg_col)
        .agg({
            "GROSS_PROFIT":"mean",
            "GROSS_MARGIN":"mean",
            "NPS RATING":"mean"
        })
        .reset_index()
    )

    aggregated[seg_name] = grouped

# -----------------------------------------
# Default view
# -----------------------------------------
default_segment = "Industry Sector"
default_metric = "Gross Profit"

default_df = aggregated[default_segment]

fig = go.Figure()

fig.add_trace(

    go.Bar(

        x=default_df["SECTOR"],
        y=default_df[metrics[default_metric]],

        text=round(default_df[metrics[default_metric]],2),
        textposition="outside",

        hovertemplate=
        "<b>%{x}</b><br>"
        + default_metric
        + ": %{y:.2f}<extra></extra>"

    )

)

# -----------------------------------------
# Create update menus
# -----------------------------------------

buttons = []

for seg_name, seg_col in segmentations.items():

    grouped = aggregated[seg_name]

    submenu = []

    for metric_name, metric_col in metrics.items():

        submenu.append(

            dict(

                label=metric_name,

                method="update",

                args=[

                    {

                        "x":[grouped[seg_col]],

                        "y":[grouped[metric_col]],

                        "text":[round(grouped[metric_col],2)]

                    },

                    {

                        "title":f"{metric_name} by {seg_name}",

                        "xaxis":{"title":seg_name},

                        "yaxis":{"title":metric_name}

                    }

                ]

            )

        )

    buttons.append(

        dict(

            label=seg_name,

            method="animate",

            args=[None],

            execute=False

        )

    )

# -----------------------------------------
# Build frames
# -----------------------------------------

frames = []

for seg_name, seg_col in segmentations.items():

    grouped = aggregated[seg_name]

    for metric_name, metric_col in metrics.items():

        frame_name = f"{seg_name}-{metric_name}"

        frames.append(

            go.Frame(

                name=frame_name,

                data=[

                    go.Bar(

                        x=grouped[seg_col],

                        y=grouped[metric_col],

                        text=round(grouped[metric_col],2),

                        textposition="outside"

                    )

                ],

                layout=go.Layout(

                    title=f"{metric_name} by {seg_name}",

                    xaxis=dict(title=seg_name),

                    yaxis=dict(title=metric_name)

                )

            )

        )

fig.frames = frames

# -----------------------------------------
# Segment dropdown
# -----------------------------------------

segment_buttons = []

for seg_name in segmentations:

    segment_buttons.append(

        dict(

            label=seg_name,

            method="animate",

            args=[[f"{seg_name}-{default_metric}"],
                  {"mode":"immediate"}]

        )

    )

# -----------------------------------------
# Metric buttons
# -----------------------------------------

metric_buttons = []

for metric_name in metrics:

    metric_buttons.append(

        dict(

            label=metric_name,

            method="animate",

            args=[[f"{default_segment}-{metric_name}"],
                  {"mode":"immediate"}]

        )

    )

# -----------------------------------------
# Layout
# -----------------------------------------

fig.update_layout(

    title="Gross Profit by Industry Sector",

    template="plotly_white",

    xaxis_title="Industry Sector",

    yaxis_title="Gross Profit",

    updatemenus=[

        dict(

            buttons=segment_buttons,

            direction="down",

            x=0,

            y=1.18,

            showactive=True

        ),

        dict(

            buttons=metric_buttons,

            direction="right",

            x=0.42,

            y=1.18,

            showactive=True

        )

    ]

)

fig.show()